# Data Conversion of SeaFET V2 .hex data
This python notebook shows how to convert a .hex file into pandas dataframe with scientific values.

Please contact [SBS customer support](https://www.seabird.com/support) for help or to request additional features.

## Init

The initialization code section below is used to import required libraries.

In [ ]:
# Native imports
from pathlib import Path

# Third-party imports
import numpy as np
import xarray as xr

# Sea-Bird imports
import seabirdscientific.conversion as sc
import seabirdscientific.instrument_data as si
import seabirdscientific.visualization as sv

# relative path imports
import example_data.example_coefficients as ec

## [Data Conversion](#proc-list)

This section shows how to convert raw data contained in a .hex file into scientific units for the instruments that follow:  
- SeaFET

In [ ]:
hex_file = Path("example_data/SeaFET_V2/SeaFET_V2.hex")

# Convert raw hexadecimal string to raw frequencies
raw_data = si.read_hex_file(
    filepath=hex_file, instrument_type=si.InstrumentType.SeaFET2, is_shallow=True
)

In [ ]:
# Convert raw frequencies to scientific values

# Salinity and pressure data used in external pH calculation
# If you have salinity and pressure data from another sensor, input it here. Otherwise adjust these reference values.
salinity = np.full(len(raw_data["scan"]), 30)  # PSU
pressure = np.full(len(raw_data["scan"]), 0)  # dbar

temperature = sc.convert_seafet_temperature(
    raw_temp=raw_data["ph temperature"].values, coefs=ec.temeprature_seafet_coefs
)

ph_internal = sc.convert_internal_seafet_ph(
    raw_ph=raw_data["vrs internal"].values, temperature=temperature, coefs=ec.ph_internal_coefs
)

ph_external = sc.convert_external_seafet_ph(
    raw_ph=raw_data["vrs external"].values,
    temperature=temperature,
    salinity=salinity,
    pressure=pressure,
    coefs=ec.ph_external_coefs,
)

# Flag to be used in data processing
flag = np.zeros(len(temperature))

dataset = xr.Dataset(
    coords={"scan": np.arange(len(temperature))},
    data_vars={
        "ph_internal": ("scan", ph_internal),
        "ph_external": ("scan", ph_external),
        "date_time": ("scan", raw_data["date time"].values),
        "flag": ("scan", flag),
    },
    attrs={"file_name": "SeaFET_V2.hex"},
)

dataset

## [Data Plotting](#proc-list)

In [ ]:
config = sv.ChartConfig(
    title="SeaFET Data Conversion",
    x_names=["date_time"],
    y_names=["ph_internal", "ph_external"],
    z_names=[],
    chart_type="overlay",
    plot_loop_edit_flags=False,
    lift_pen_over_bad_data=True,
)

fig = sv.plot_xy_chart(dataset, config)

# plotly customizations
fig["layout"]["yaxis"]["autorange"] = "reversed"
fig.data[0].name = "pH Internal"
fig.data[1].name = "pH External"
fig["layout"]["yaxis"]["title"] = "pH Internal"
fig["layout"]["yaxis2"]["title"] = "pH External"


fig.update_layout(height=800)
fig.show()

## Processing

This converted data should now be processed using the tools in processing.ipynb to produce a final data product.